<a href="https://colab.research.google.com/github/Thawin2551/python/blob/main/600636-OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 600636 | นายธาวิน ชาวหวายสอ

# เทคนิคที่ใช้ในการทำ OCR

<table border="1" style="border-collapse: collapse; width: 100%; text-align: left; font-family: sans-serif;">
  <thead>
    <tr style="background-color: #e6f7ff;">
      <th style="padding: 10px; width: 25%;">Category</th>
      <th style="padding: 10px; width: 30%;">Key Technique</th>
      <th style="padding: 10px; width: 45%;">Executive Summary</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 10px;"><b>1. Image Preprocessing</b></td>
      <td style="padding: 10px;"><b>Stitching & Sharpening</b><br>ต่อภาพ + ดึงความคมชัด</td>
      <td style="padding: 10px;">ต่อเอกสารหลายหน้าเป็นรูปเดียว (Vertical Stack) ให้ AI มองเห็นบริบททั้งตารางอย่างครบถ้วน และใช้ OpenCV (Sharpening Kernel) เพิ่มความคมชัด ลดข้อผิดพลาดจากภาพถ่ายเบลอ</td>
    </tr>
    <tr>
      <td style="padding: 10px;"><b>2. Prompt Engineering</b></td>
      <td style="padding: 10px;"><b>Constrained JSON Output</b><br>บังคับโครงสร้างข้อมูล</td>
      <td style="padding: 10px;">เขียน Prompt ให้ดึงเฉพาะ "เบอร์" คู่กับ "คะแนน" ตามด้วยเทคนิค <b>บังคับ Output เป็น JSON 100%</td>
    </tr>
    <tr>
      <td style="padding: 10px;"><b>3. Data Cleaning & Validation</b></td>
      <td style="padding: 10px;"><b>Cross-Check ด้วยคำอ่าน<br>และแปลงเลขไทย</b></td>
      <td style="padding: 10px;">ทลายขีดจำกัด AI ด้วยการ <b>แปลงเลขไทยเป็นอารบิกอัตโนมัติ</b> และสกัด <b>"คำอ่านในวงเล็บ"</b> มารีเวิร์สเป็นตัวเลข พร้อม Regex กรองขยะ ส.ส. หรือรอยขีดทิ้ง</td>
    </tr>
    <tr>
      <td style="padding: 10px;"><b>4. Pipeline Resiliency</b></td>
      <td style="padding: 10px;"><b>Auto-Save (Caching) &<br>Rate Limit Fallback</b></td>
      <td style="padding: 10px;">ออกแบบระบบให้มี <b>Auto-Save</b> บันทึกผลแบบเรียลไทม์ หากเจอปัญหาโควต้าฟรี (Rate Limit) หรือเน็ตตัด โปรแกรมสามารถดีเลย์ตัวเอง และ <b>รันต่อจากเดิมได้เลย</b> ไม่ต้องเริ่มใหม่ให้เสียเวลาและติด limit</td>
    </tr>
  </tbody>
</table>


# Gemini.md (Description)

ผมได้ให้ gemini เก็บข้อมูลและบันทึกตลอดการทำ ocr ว่าทำอะไรไปบ้างผ่านไฟล์ gemini.md

# บันทึกการทำงาน (Gemini.md)

## เริ่มต้นการปรับปรุงระบบ OCR โดยใช้ Gemini API Key
- ได้รับ API Key จากผู้ใช้ (`##########`)
- กำลังเตรียมปรับเปลี่ยนโค้ดจากการใช้ Vertex AI (`google-cloud-aiplatform`) มาเป็นใช้ `google-generativeai` ซึ่งรองรับ API Key โดยตรง
- อัปเดต `config.py` ให้ใช้ `API_KEY` แทน `PROJECT_ID` และ `LOCATION`
- อัปเดต `ocr_engines.py` เพื่อใช้ `google.generativeai` แทน `vertexai`
- อัปเดต `main.py` ทำการเรียก `init_gemini()`
- ทำการติดตั้งแพ็กเกจ `google-generativeai` ผ่าน pip

## การปรับปรุงระบบอ่านคะแนนผู้สมัครแบบกลุ่ม (Multi-Candidate per Document)
- พบว่าใน 1 ภาพเอกสาร (เช่น `constituency_10_1.png`) มีตารางคะแนนผู้สมัครหลายเบอร์พร้อมกัน
- **อัปเดต `ocr_engines.py`**: แก้ไข Prompt ให้ AI ดึงข้อมูล "เบอร์ผู้สมัคร" และ "คะแนน" ของทุกคนในภาพออกมาเป็น JSON Dictionary รวดเดียว (`{"1": 150, "2": 20}`)
- **อัปเดต `utils.py`**: เพิ่มระบบจัดเรียงภาพ (`page1, page2, page10`) ให้ถูกต้องก่อนนำมาต่อกัน และเปลี่ยนระบบตรวจสอบความถูกต้องให้รองรับ Dictionary Output
- **อัปเดต `main.py`**:
  - เปลี่ยนจากการไล่ตาม CSV ID เป็นการหาไฟล์ภาพที่มีอยู่จริง นำมาจับกลุ่ม (Group by document name)
  - หลังจาก OCR แล้ว จะทำการ Export ข้อมูลดิบเป็นไฟล์ `extracted_formatted.json` โดยมีเงื่อนไขการเรียงลำดับใหม่:
    - `constituency` เรียงตามคะแนน (`votes`) จากมากไปน้อย
    - `party_list` เรียงตามเบอร์ผู้สมัคร (`candidate` number) จากน้อยไปมาก
  - ท้ายสุดทำการแมปผลลัพธ์จาก Cache กลับไปยัง Template CSV (`submission_ocr_v4.csv`) ให้ถูกต้องตรงตาม ID ทุกบรรทัด

## เปลี่ยนการใช้โมเดลและรันต่อจาก Cache
- ผู้ใช้แจ้งว่า `gemini-2.5-flash` ติด Rate Limit จึงแก้ไข `config.py` ให้เป็น `gemini-2.5-pro`
- แก้ไข `main.py` เพื่อให้รันต่อจากไฟล์ `extracted_ocr.json` (สำหรับ Object ไหนที่เป็น `{}` ว่างๆ ซึ่งเกิดจาก Error/Rate Limit ก่อนหน้านี้ จะยอมให้ระบบรันใหม่)
- ทำการสั่งรัน pipeline ต่อจนจบ

## การเปลี่ยนกลับไปใช้ Vertex AI เพื่อดึงเครดิตฟรี (Express API)
- ผู้ใช้สร้าง API Key ใหม่ที่ผูกกับ Vertex AI เพื่อใช้เครดิต $300 แทน
- ปรับแต่ง `ocr_engines.py` ใหม่ โดยยกเลิกการใช้ SDK ของ `google-generativeai` ชั่วคราว และเขียนแบบยิงตรงผ่าน `requests.post()` ไปยัง Endpoint `aiplatform.googleapis.com` แทน
- พบปัญหา **429 Resource Exhausted** เนื่องจากโปรแกรมรันส่งภาพต่อเนื่องเร็วเกินไป จนชน Rate Limit ของ Vertex AI
- เพิ่มระบบ **Auto-Retry** เข้าไปใน `main.py`: เมื่อโปรแกรมเจอ Error 429 มันจะหยุดรอ (`time.sleep`) เป็นเวลา 30, 60, 90 วินาทีตามลำดับแล้วค่อยลองส่งข้อความใหม่ให้เอง โดยไม่ข้ามไฟล์นั้นไป
- **อัปเดตโมเดลเป็น `gemini-3-flash`**: เนื่องจาก Vertex AI แบบ Free Tier จำกัด Quota ต่อวันเพียง 20 Requests ต่อโมเดล จึงต้องหันมาใช้ `gemini-3-flash` แทนเพื่อทำงานที่ค้างอยู่ต่อ


# Model and Text-Editor

ผมใช้ Antigravity และใช้ model gemini 2.5 flash ในการทำ ocr โดยใช้เครดิตของ vertex AI ที่แจกฟรีมาใช้ในการเพิ่ม request ต่อวัน ครับ

# config.py | สำหรับการตั้งค่าต่าง ๆ

In [ ]:
# config.py
API_KEY = "########"
MODEL_NAME = "gemini-2.5-flash"

INPUT_DIR = r"C:\Users\tumpa\OneDrive\Desktop\SuperAI\images"
TEMPLATE_PATH = r"C:\Users\tumpa\OneDrive\Desktop\SuperAI\submission_template_v4.csv"
CACHE_FILE = r"C:\Users\tumpa\OneDrive\Desktop\SuperAI\ocr_cache_vertex.json"
FINAL_OUTPUT = r"C:\Users\tumpa\OneDrive\Desktop\SuperAI\submission_ocr_v4.csv"

PREPROCESS_RESIZE_WIDTH = 1000
PREPROCESS_SHARPEN = True


# utils.py | ไว้ทำการ map ตัวเลขไทยให้เป็นตัวเลขอารบิกรวมถึงให้อ่านตัวอ่านที่อยู่วงเล็บหลังเลขไทย เช่น หนึ่งร้อย

In [ ]:
# utils.py
import re
import cv2
import numpy as np

THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
# คำอ่านตัวเลขแบบเรียงตัวและแบบหลัก
THAI_NUM_WORDS = {
    'ศูนย์': '0', 'หนึ่ง': '1', 'เอ็ด': '1', 'สอง': '2', 'ยี่': '2',
    'สาม': '3', 'สี่': '4', 'ห้า': '5', 'หก': '6', 'เจ็ด': '7', 'แปด': '8', 'เก้า': '9'
}

def parse_thai_words_to_digits(text):
    """แปลงคำอ่านภาษาไทยแบบเรียงตัว เช่น 'หนึ่งสองสอง' ให้เป็น '122'"""
    # กรองเฉพาะคำที่เป็นตัวเลข
    pattern = re.compile(r'(ศูนย์|หนึ่ง|เอ็ด|สอง|ยี่|สาม|สี่|ห้า|หก|เจ็ด|แปด|เก้า)')
    matches = pattern.findall(text)

    if matches:
        return "".join([THAI_NUM_WORDS[m] for m in matches])
    return None

def extract_page_number(path):
    path_lower = path.lower()
    match = re.search(r'page(\d+)', path_lower)
    if match:
        return int(match.group(1))
    return 0

def preprocess_images(image_paths, target_width=1000, apply_sharpen=True):
    """
    โหลดภาพหลายหน้า, จัดเรียงตาม page, ปรับขนาดให้กว้างเท่ากัน, นำมาต่อกัน (Vertical Stack), และทำ Sharpening
    รีเทิร์นเป็น image bytes สำหรับส่งให้ Gemini
    """
    # เรียงลำดับไฟล์ตามเลขหน้าให้ถูกต้อง
    image_paths = sorted(image_paths, key=extract_page_number)

    images = []
    for path in image_paths:
        img = cv2.imread(path)
        if img is None:
            continue

        # ปรับขนาดภาพให้มีความกว้างเท่ากับ target_width เพื่อให้ต่อกันได้พอดี
        h, w = img.shape[:2]
        if w != target_width:
            ratio = target_width / w
            new_h = int(h * ratio)
            img = cv2.resize(img, (target_width, new_h), interpolation=cv2.INTER_AREA)

        images.append(img)

    if not images:
        return None

    # นำภาพมาต่อกันในแนวตั้ง
    combined_img = np.vstack(images)

    # เพิ่มความคมชัด (Sharpening) เพื่อให้อ่านตัวเลขอารบิกและไทยได้ชัดขึ้น
    if apply_sharpen:
        kernel = np.array([[-1, -1, -1],
                           [-1,  9, -1],
                           [-1, -1, -1]])
        combined_img = cv2.filter2D(combined_img, -1, kernel)

    # แปลงกลับเป็น Bytes เพื่อส่งเข้า API
    success, encoded_image = cv2.imencode('.jpg', combined_img)
    if success:
        return encoded_image.tobytes()
    return None

def clean_and_verify_dict(raw_json_dict):
    """
    รับค่าจาก OCR ที่เป็น dictionary (key เป็นเบอร์, value เป็นคะแนน)
    แล้วทำความสะอาด แปลงไทยเป็นอารบิก แยกวงเล็บ
    return เป็น dict ที่ clean แล้ว: { "1": 120, "2": 50 }
    """
    cleaned = {}
    if not isinstance(raw_json_dict, dict):
        return cleaned

    for k, v in raw_json_dict.items():
        text = str(v).replace(',', '').replace(' ', '')
        text_arabic = text.translate(THAI_DIGITS)

        # ลบตัวอักษรขยะหรือเลขหน้าที่ติดมา
        text_arabic = re.sub(r'(6/1|5/18|page.*|บช|ส\.ส\.)', '', text_arabic, flags=re.IGNORECASE)

        num_match = re.search(r'(\d+)', text_arabic)
        main_num = int(num_match.group(1)) if num_match else 0

        # ตรวจสอบตัวเลขในวงเล็บ (ถ้ามี) แบบเรียงตัว เช่น (หนึ่งสองสอง)
        word_match = re.search(r'\((.*?)\)', text_arabic)
        if word_match:
            word_digits = parse_thai_words_to_digits(word_match.group(1))
            if word_digits and int(word_digits) != main_num:
                main_num = int(word_digits)

        # ใช้ Key เดิม (เบอร์ผู้สมัคร) ขจัดช่องว่าง
        cleaned[str(k).strip()] = main_num

    return cleaned

# ocr_engines.py | ทำการป้อน prompt ให้ gemini 2.5 flash ทำ ocr เอกสารทั้งหมด

In [ ]:
# ocr_engines.py
import requests
import base64
import config

def init_gemini():
    # ไม่ต้องเซ็ตอัพ SDK แค่ส่งกลับชื่อ Model ที่จะใช้ใน Endpoint URL
    return config.MODEL_NAME

def call_gemini_ocr(model_name, image_bytes):
    # แปลงภาพเป็น base64
    encoded_image = base64.b64encode(image_bytes).decode('utf-8')

    # เช็คว่าใช้ API Key ของ AI Studio หรือ Vertex AI
    if config.API_KEY.startswith("AIza"):
        # Google AI Studio endpoint (1,500 requests/day free)
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={config.API_KEY}"
    else:
        # Vertex AI Express API endpoint (20 requests/day per model on free tier)
        url = f"https://aiplatform.googleapis.com/v1/publishers/google/models/{model_name}:generateContent?key={config.API_KEY}"


    prompt = """คุณคือผู้เชี่ยวชาญด้าน OCR สำหรับเอกสารการเลือกตั้ง
    รูปภาพนี้คือตารางคะแนนผู้สมัคร (อาจมีหลายแผ่นต่อกัน)

    กฎในการอ่าน:
    1. ค้นหา "หมายเลขผู้สมัคร" (เบอร์) และ "คะแนน" (หรือจำนวนบัตร) ของผู้สมัครแต่ละคน
    2. อ่านคะแนนให้ครอบคลุมทั้งตัวเลขอารบิกและเลขไทย (เช่น ๑๒๓)
    3. หากมีคำอ่านภาษาไทยในวงเล็บหลังตัวเลข (เช่น ๑๒๒ (หนึ่งสองสอง)) ให้ดึงมาด้วยทั้งหมดในเครื่องหมายคำพูด
    4. ตัด comma (,) ออกให้หมด
    5. ไม่สนใจหัวกระดาษหรือตัวเลขที่ไม่เกี่ยวกับคะแนนผู้สมัคร

    ตอบกลับในรูปแบบ JSON dictionary เท่านั้น โดยเอา "เบอร์ผู้สมัคร" (เป็น string) เป็น key และ "คะแนน" เป็น value
    ตัวอย่าง:
    {
        "1": "122 (หนึ่งสองสอง)",
        "2": "50",
        "3": "0"
    }"""

    payload = {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {
                        "text": prompt
                    },
                    {
                        "inlineData": {
                            "mimeType": "image/jpeg",
                            "data": encoded_image
                        }
                    }
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0.0,
            "responseMimeType": "application/json"
        }
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Vertex API Error: {response.text}")

    res_json = response.json()

    try:
        text = res_json['candidates'][0]['content']['parts'][0]['text']
        return text
    except (KeyError, IndexError):
        raise Exception(f"Unexpected response format: {res_json}")

# main.py | หลังจากการสร้างไฟล์ทั้งหมดเราก็ Run Code ผ่านไฟล์ main.py

โดยเมื่อรันเสร็จเราจะ Extracted File กลับเข้าไปใน extracted_ocr.json ในรูปนี้พรรคไหนได้ votes เท่าไหร่บ้าง

In [ ]:
# main.py
import os
import glob
import json
import re
import pandas as pd
from tqdm import tqdm
import config
import utils
import ocr_engines

def get_base_id(filename):
    basename = os.path.basename(filename)
    match = re.match(r'((constituency|party_list)_\d+_\d+)', basename)
    if match:
        return match.group(1)
    return None

def main():
    model = ocr_engines.init_gemini()

    # 1. Group images by base document ID
    search_pattern = os.path.join(config.INPUT_DIR, "*.*")
    all_files = [f for f in glob.glob(search_pattern) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    doc_groups = {}
    for f in all_files:
        base_id = get_base_id(f)
        if base_id:
            if base_id not in doc_groups:
                doc_groups[base_id] = []
            doc_groups[base_id].append(f)

    # Use a new cache file for the updated dictionary-based OCR
    cache_file = os.path.join(os.path.dirname(config.CACHE_FILE), "extracted_ocr.json")
    cache = {}
    if os.path.exists(cache_file):
        with open(cache_file, 'r', encoding='utf-8') as f:
            cache = json.load(f)

        # บังคับให้ party_list ใน cache ที่โหลดมามี 57 เบอร์แน่นอน
        for k, v in cache.items():
            if k.startswith('party_list') and isinstance(v, dict) and len(v) > 0:
                enforced_dict = {}
                for i in range(1, 58):
                    key_str = str(i)
                    enforced_dict[key_str] = v.get(key_str, 0)
                cache[k] = enforced_dict

    # 2. Process each document group
    pbar = tqdm(doc_groups.items(), desc="OCR Processing")
    for base_id, image_paths in pbar:
        if base_id in cache and cache[base_id]:
            continue

        try:
            pbar.set_postfix(id=base_id, status="PREPROCESSING")
            image_bytes = utils.preprocess_images(image_paths,
                                                  target_width=config.PREPROCESS_RESIZE_WIDTH,
                                                  apply_sharpen=config.PREPROCESS_SHARPEN)

            if not image_bytes:
                cache[base_id] = {}
                continue

            pbar.set_postfix(id=base_id, status="AI_CALLING")
            raw_res = ocr_engines.call_gemini_ocr(model, image_bytes)

            raw_res = raw_res.strip()
            if raw_res.startswith("```json"):
                raw_res = raw_res[7:-3].strip()
            elif raw_res.startswith("```"):
                raw_res = raw_res[3:-3].strip()

            res_json = json.loads(raw_res)

            if isinstance(res_json, dict) and "candidates" in res_json:
                res_dict = res_json["candidates"]
            else:
                res_dict = res_json

            cleaned_dict = utils.clean_and_verify_dict(res_dict)

            # บังคับให้ party_list ที่แกะจาก OCR ใหม่มี 57 เบอร์เสมอ
            if base_id.startswith('party_list'):
                enforced_dict = {}
                for i in range(1, 58):
                    key_str = str(i)
                    enforced_dict[key_str] = cleaned_dict.get(key_str, 0)
                cleaned_dict = enforced_dict

            cache[base_id] = cleaned_dict

            if len(cache) % 5 == 0:
                with open(cache_file, 'w', encoding='utf-8') as f:
                    json.dump(cache, f, ensure_ascii=False, indent=4)

        except Exception as e:
            print(f"\\nError on {base_id}: {e}")
            cache[base_id] = {}

    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=4)

    # 3. Format exactly as user requested (Sorted JSON)
    formatted_output = {}
    for base_id, candidates_dict in cache.items():
        cand_list = []
        for cand, votes in candidates_dict.items():
            try:
                cand_list.append({"candidate": int(cand), "votes": int(votes)})
            except ValueError:
                pass

        # เรียงลำดับตามเงื่อนไขของผู้ใช้
        if base_id.startswith('constituency'):
            cand_list.sort(key=lambda x: x["votes"], reverse=True)
        else:
            cand_list.sort(key=lambda x: x["candidate"])

        formatted_output[base_id] = cand_list

    formatted_json_path = os.path.join(os.path.dirname(config.CACHE_FILE), "extracted_formatted.json")
    with open(formatted_json_path, 'w', encoding='utf-8') as f:
        json.dump(formatted_output, f, ensure_ascii=False, indent=4)
    print(f"\\nSaved customized formatted JSON to {formatted_json_path}")

    # 4. Generate the final Submission CSV mapping to exact IDs
    if os.path.exists(config.TEMPLATE_PATH):
        df_template = pd.read_csv(config.TEMPLATE_PATH)
        df_final = df_template.copy()

        final_votes = []
        for tid in df_final['id']:
            match = re.match(r'((constituency|party_list)_\d+_\d+)_(\d+)', str(tid))
            if match:
                base_id = match.group(1)
                cand_num = str(int(match.group(3)))

                if base_id in cache and cand_num in cache[base_id]:
                    final_votes.append(cache[base_id][cand_num])
                else:
                    final_votes.append(0)
            else:
                final_votes.append(0)

        df_final['votes'] = final_votes
        df_final.to_csv(config.FINAL_OUTPUT, index=False)
        print(f"Done! Saved CSV to {config.FINAL_OUTPUT}")

if __name__ == "__main__":
    main()

# import json, io | เพื่อแสดงไฟล์ตัวอย่างที่ extracted ออกมาจากเอกสารให้อยู่ในรูปแบบของ json และมีการเก็บ cache ไว้เพื่อที่กลับมาเทรนต่อหรือ api limit exceed จะได้ไม่ต้องเทรนใหม่ครับ

In [ ]:
import json
import io

In [ ]:
import json
import io

# Read the content and load it as a Python dictionary
with open('/content/extracted_ocr.json', 'r', encoding='utf-8') as f:
    extracted_ocr__json = json.load(f)

with open('/content/extracted_formatted.json', 'r', encoding='utf-8') as f:
    extracted_formatted__json = json.load(f)

print("Extracted OCR JSON:")
print(extracted_ocr__json)
print("\nExtracted Formatted JSON:")
print(extracted_formatted__json)

Extracted OCR JSON:
{'constituency_10_1': {'5': 3167, '8': 183, '1': 14368, '4': 63, '6': 275, '12': 1133, '7': 123, '2': 979, '11': 629, '16': 489, '15': 353, '3': 288, '10': 165, '13': 155, '14': 13, '18': 44, '17': 8}, 'constituency_10_10': {'6': 4184, '5': 1987, '9': 944, '13': 7265, '2': 6332, '11': 22, '8': 1487, '14': 636, '12': 543, '16': 545, '4': 53, '7': 495, '10': 461, '3': 282, '15': 23, '17': 168}, 'constituency_10_11': {'1': 37878, '7': 28842, '13': 2844, '14': 3832, '9': 178, '5': 556, '12': 556, '6': 682, '11': 62, '17': 584, '4': 483, '16': 367, '2': 322, '3': 182, '8': 23, '10': 184, '15': 63}, 'constituency_10_12': {'15': 49925, '13': 1616, '12': 1545, '6': 9877, '9': 4442, '11': 7878, '14': 777, '8': 7833, '16': 999, '2': 776, '5': 452, '4': 43, '1': 434, '3': 368, '7': 298, '10': 267}, 'constituency_10_13': {'3': 337, '4': 324, '2': 1742, '11': 7887, '9': 7343, '1': 7343, '6': 182, '14': 87, '5': 334, '12': 222, '15': 197, '16': 183}, 'constituency_10_14': {'14': 

 # Submission File | หลังจากที่ extracted ใส่ json file เราจะเอามารวมกับ template และได้เป็น file submission ออกมา

In [ ]:
import pandas as pd

In [ ]:
df_template = pd.read_csv("/content/submission_template_v4.csv")
df_submission = pd.read_csv("/content/submission_ocr_v4.csv")

In [ ]:
print("Template")
df_template

Template


,id,votes
0,constituency_10_1_1,0
1,constituency_10_1_2,0
2,constituency_10_1_3,0
3,constituency_10_1_4,0
4,constituency_10_1_5,0
...,...,...
10048,party_list_34_11_53,0
10049,party_list_34_11_54,0
10050,party_list_34_11_55,0
10051,party_list_34_11_56,0


ไฟล์ submssion

In [ ]:
print("Submission File")
df_submission

Submission File


,id,votes
0,constituency_10_1_1,14368
1,constituency_10_1_2,979
2,constituency_10_1_3,288
3,constituency_10_1_4,63
4,constituency_10_1_5,3167
...,...,...
10048,party_list_34_11_53,14
10049,party_list_34_11_54,41
10050,party_list_34_11_55,29
10051,party_list_34_11_56,43
